In [0]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType
from datetime import datetime
from zoneinfo import ZoneInfo
import time

start_time = time.time()

# Initialize Spark Session (already available in Databricks as 'spark')
# spark = SparkSession.builder.getOrCreate()

# Invoice table shared by finance
invoices_df = spark.read.table("furlenco_analytics.user_uploads.invoice_equi")
vertical = invoices_df.select("vertical").first()[0].lower()

# Select and rename columns
invoices_df = invoices_df.select(
    F.col("fur_id").alias("furid"),
    F.col("Recognised date").alias("invoice_date"),
    F.col("Total").alias("invoice_value")
)

# Closing Balance Table shared by finance
balance_df = spark.read.table("furlenco_analytics.user_uploads.unlmtd_tr_bal_30th_sep_25_given_to_auditors")

# Date to be modified as required
current_date = datetime(2025, 9, 30).date()

# Age bucket definitions
AGE_LABELS = [
    'ZeroTo30',
    'ThirtyOneToSixty',
    'SixtyOneToNinety',
    'NinetyOneToOneTwenty',
    'OneTwentyOneToOneEighty',
    'MoreThanOneEighty'
]

# --- 2. Data Cleaning and Preparation ---
# Clean balance data
balance_df = balance_df.toDF("furid", "balance")
balance_df = balance_df.filter(
    (F.col("furid").isNotNull()) & 
    (F.lower(F.col("furid")) != "total")
)

# Clean balance values - remove commas and convert to double
balance_df = balance_df.withColumn(
    "balance",
    F.regexp_replace(F.col("balance").cast(StringType()), ",", "").cast(DoubleType())
)

# Filter users with outstanding balance
users_with_outstanding = balance_df.filter(F.col("balance") > 0)

# Clean invoices data
invoices_df = invoices_df.filter(F.col("furid").isNotNull())

# Clean invoice values - remove commas and convert to double
invoices_df = invoices_df.withColumn(
    "invoice_value",
    F.regexp_replace(F.col("invoice_value").cast(StringType()), ",", "").cast(DoubleType())
)
invoices_df = invoices_df.filter(F.col("invoice_value") >= 0)

# Convert invoice dates (assuming DD/MM/YYYY or DD-MM-YYYY format)
invoices_df = invoices_df.withColumn(
    "invoice_date",
    F.to_date(F.col("invoice_date"), "dd/MM/yyyy")
)
invoices_df = invoices_df.filter(F.col("invoice_date").isNotNull())

# Calculate age and age bucket
invoices_df = invoices_df.withColumn(
    "age",
    F.datediff(F.lit(current_date), F.col("invoice_date"))
)

# Create age bucket using when/otherwise chain
invoices_df = invoices_df.withColumn(
    "age_bucket",
    F.when(F.col("age") <= 30, "ZeroTo30")
    .when(F.col("age") <= 60, "ThirtyOneToSixty")
    .when(F.col("age") <= 90, "SixtyOneToNinety")
    .when(F.col("age") <= 120, "NinetyOneToOneTwenty")
    .when(F.col("age") <= 180, "OneTwentyOneToOneEighty")
    .otherwise("MoreThanOneEighty")
)

# --- 3. Core Allocation Logic ---
# Merge balances with invoices
merged_df = users_with_outstanding.join(invoices_df, on="furid", how="left")

# Fill null invoice values with 0
merged_df = merged_df.fillna({"invoice_value": 0, "age": 999999})

# Define window spec for cumulative sum (ordered by age, oldest first)
window_spec = Window.partitionBy("furid").orderBy("age").rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Calculate cumulative invoice sum
merged_df = merged_df.withColumn(
    "cumulative_invoice",
    F.sum("invoice_value").over(window_spec)
)

# Calculate previous cumulative (balance used by previous invoices)
merged_df = merged_df.withColumn(
    "prev_cumulative_invoice",
    F.col("cumulative_invoice") - F.col("invoice_value")
)

# Allocate amount: min(remaining_balance, invoice_value)
merged_df = merged_df.withColumn(
    "remaining_balance",
    F.col("balance") - F.col("prev_cumulative_invoice")
)

merged_df = merged_df.withColumn(
    "amount",
    F.when(F.col("remaining_balance") <= 0, 0)
    .when(F.col("remaining_balance") >= F.col("invoice_value"), F.col("invoice_value"))
    .otherwise(F.col("remaining_balance"))
)

# Calculate residual balance per user
user_summary = merged_df.groupBy("furid").agg(
    F.sum("amount").alias("total_allocated"),
    F.first("balance").alias("balance")
)

user_summary = user_summary.withColumn(
    "residual_balance",
    F.col("balance") - F.col("total_allocated")
)

# Create residual dataframe for MoreThanOneEighty bucket
residual_df = user_summary.filter(F.col("residual_balance") > 0.01).select(
    F.col("furid"),
    F.lit("MoreThanOneEighty").alias("age_bucket"),
    F.col("residual_balance").alias("amount")
)

# --- 4. Final Aggregation and Formatting ---
# Combine primary allocations with residuals
final_report_data = merged_df.select("furid", "age_bucket", "amount").union(residual_df)

# Aggregate by furid and age_bucket
final_report_data = final_report_data.groupBy("furid", "age_bucket").agg(
    F.sum("amount").alias("amount")
)

# Pivot to create columns for each age bucket
final_report = final_report_data.groupBy("furid").pivot("age_bucket", AGE_LABELS).sum("amount")

# Fill null values with 0
for col in AGE_LABELS:
    final_report = final_report.fillna({col: 0})

# Ensure all columns exist and order them correctly
select_cols = ["furid"] + AGE_LABELS
final_report = final_report.select(*select_cols)

# Rename furid to FurID
final_report = final_report.withColumnRenamed("furid", "FurID")

# --- 5. Save Output ---
output_filename = f'invoicing_ageing_report_{datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%Y-%m-%dT%H:%M")}_{vertical}_by_invoice_age.csv'

display(final_report)
# Write to CSV (single file)
# final_report.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"/tmp/{output_filename}")

# If you want to save to DBFS or download locally, you can also use:
# final_report.toPandas().to_csv(output_filename, index=False)

end_time = time.time()

print(f"✅ Report generation complete!")
print(f"   Saved to: {output_filename}")
print(f"   Execution time: {end_time - start_time:.4f} seconds.")
print(f"   Total records: {final_report.count()}")